### Mission Objective

Below, I will provide an example of **character-level text generation implemented from scratch**. The logic is as follows:

* **Input a short text** to serve as training data.
* **At each step, have the RNN look at the previous character** and predict the next character.

In [1]:
import numpy as np

In [2]:
text = "i like machine learning . i like deep learning . i like neural networks ."

tokens = text.split()
print(tokens)

['i', 'like', 'machine', 'learning', '.', 'i', 'like', 'deep', 'learning', '.', 'i', 'like', 'neural', 'networks', '.']


In [3]:
vocab = sorted(list(set(tokens)))
vocab_size = len(vocab)

word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

print("词表：", vocab)
print("词表大小：", vocab_size)

词表： ['.', 'deep', 'i', 'learning', 'like', 'machine', 'networks', 'neural']
词表大小： 8


In [4]:
def one_hot(index, vocab_size):
    x = np.zeros((vocab_size, 1))
    x[index] = 1
    return x

In [5]:
print("词 like 的索引:", word_to_idx["like"])
print(one_hot(word_to_idx["like"], vocab_size).T)

词 like 的索引: 4
[[0. 0. 0. 0. 1. 0. 0. 0.]]


In [6]:
def initialize_parameters(n_a, n_x, n_y):
    np.random.seed(1)
    Wax = np.random.randn(n_a, n_x) * 0.01
    Waa = np.random.randn(n_a, n_a) * 0.01
    Wya = np.random.randn(n_y, n_a) * 0.01
    ba = np.zeros((n_a, 1))
    by = np.zeros((n_y, 1))
    
    parameters = {
        "Wax": Wax,
        "Waa": Waa,
        "Wya": Wya,
        "ba": ba,
        "by": by
    }
    return parameters

In [7]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / np.sum(e_x, axis=0, keepdims=True)

In [8]:
def rnn_cell_forward(xt, a_prev, parameters):
    Wax = parameters["Wax"]
    Waa = parameters["Waa"]
    Wya = parameters["Wya"]
    ba = parameters["ba"]
    by = parameters["by"]
    
    a_next = np.tanh(np.dot(Wax, xt) + np.dot(Waa, a_prev) + ba)
    yt_pred = softmax(np.dot(Wya, a_next) + by)
    
    cache = (a_next, a_prev, xt, parameters)
    return a_next, yt_pred, cache

In [9]:
def rnn_forward(X, Y, a0, parameters, vocab_size):
    x, a, y_pred = {}, {}, {}
    caches = []
    loss = 0.0
    
    a[-1] = np.copy(a0)
    
    for t in range(len(X)):
        x[t] = one_hot(X[t], vocab_size)
        a[t], y_pred[t], cache = rnn_cell_forward(x[t], a[t-1], parameters)
        caches.append(cache)
        
        loss_t = -np.log(y_pred[t][Y[t], 0] + 1e-12)
        loss += loss_t
    
    return loss, (caches, x, a, y_pred)

In [10]:
def rnn_cell_backward(dy, gradients, cache):
    (a_next, a_prev, xt, parameters) = cache
    
    Wax = parameters["Wax"]
    Waa = parameters["Waa"]
    Wya = parameters["Wya"]
    
    gradients["dWya"] += np.dot(dy, a_next.T)
    gradients["dby"] += dy
    
    da = np.dot(Wya.T, dy) + gradients["da_next"]
    dtanh = (1 - a_next**2) * da
    
    gradients["dba"] += dtanh
    gradients["dWax"] += np.dot(dtanh, xt.T)
    gradients["dWaa"] += np.dot(dtanh, a_prev.T)
    gradients["da_next"] = np.dot(Waa.T, dtanh)
    
    return gradients

In [11]:
def rnn_backward(X, Y, parameters, cache_tuple, vocab_size):
    (caches, x, a, y_pred) = cache_tuple
    
    n_a = parameters["Waa"].shape[0]
    
    gradients = {
        "dWax": np.zeros_like(parameters["Wax"]),
        "dWaa": np.zeros_like(parameters["Waa"]),
        "dWya": np.zeros_like(parameters["Wya"]),
        "dba": np.zeros_like(parameters["ba"]),
        "dby": np.zeros_like(parameters["by"]),
        "da_next": np.zeros((n_a, 1))
    }
    
    for t in reversed(range(len(X))):
        dy = np.copy(y_pred[t])
        dy[Y[t]] -= 1
        gradients = rnn_cell_backward(dy, gradients, caches[t])
    
    return gradients, a[len(X)-1]

In [12]:
def clip(gradients, max_value):
    for key in ["dWax", "dWaa", "dWya", "dba", "dby"]:
        np.clip(gradients[key], -max_value, max_value, out=gradients[key])
    return gradients

In [13]:
def update_parameters(parameters, gradients, lr):
    parameters["Wax"] -= lr * gradients["dWax"]
    parameters["Waa"] -= lr * gradients["dWaa"]
    parameters["Wya"] -= lr * gradients["dWya"]
    parameters["ba"]  -= lr * gradients["dba"]
    parameters["by"]  -= lr * gradients["dby"]
    return parameters

In [14]:
tokens = ['i', 'like', 'machine', 'learning', '.']

In [15]:
X = [word_to_idx[w] for w in tokens[:-1]]
Y = [word_to_idx[w] for w in tokens[1:]]

print("X =", [idx_to_word[i] for i in X])
print("Y =", [idx_to_word[i] for i in Y])

X = ['i', 'like', 'machine', 'learning']
Y = ['like', 'machine', 'learning', '.']


In [16]:
def get_top_k_probs(y_pred, idx_to_word, top_k=3):
    probs = y_pred.ravel()
    top_indices = np.argsort(probs)[::-1][:top_k]
    result = []
    for idx in top_indices:
        result.append((idx_to_word[idx], float(probs[idx])))
    return result

In [17]:
def generate_text_with_probabilities(parameters, word_to_idx, idx_to_word,
                                     seed_word="i", length=8, top_k=3):
    vocab_size = len(word_to_idx)
    n_a = parameters["Waa"].shape[0]
    
    x = one_hot(word_to_idx[seed_word], vocab_size)
    a_prev = np.zeros((n_a, 1))
    
    generated_words = [seed_word]
    details = []
    
    current_word = seed_word
    
    for step in range(length - 1):
        a_prev, y_pred, _ = rnn_cell_forward(x, a_prev, parameters)
        
        probs = y_pred.ravel()
        next_idx = np.random.choice(range(vocab_size), p=probs)
        next_word = idx_to_word[next_idx]
        next_prob = float(probs[next_idx])
        
        top_candidates = get_top_k_probs(y_pred, idx_to_word, top_k=top_k)
        
        details.append({
            "step": step + 1,
            "input_word": current_word,
            "predicted_word": next_word,
            "predicted_prob": next_prob,
            "top_candidates": top_candidates
        })
        
        generated_words.append(next_word)
        current_word = next_word
        x = one_hot(next_idx, vocab_size)
    
    return " ".join(generated_words), details

In [18]:
def train_rnn_word_level(tokens, n_a=32, lr=0.1, num_iterations=2000, print_every=200):
    vocab = sorted(list(set(tokens)))
    vocab_size = len(vocab)

    word_to_idx = {w: i for i, w in enumerate(vocab)}
    idx_to_word = {i: w for w, i in word_to_idx.items()}
    
    X = [word_to_idx[w] for w in tokens[:-1]]
    Y = [word_to_idx[w] for w in tokens[1:]]
    
    parameters = initialize_parameters(n_a, vocab_size, vocab_size)
    a_prev = np.zeros((n_a, 1))
    
    for i in range(num_iterations):
        loss, cache_tuple = rnn_forward(X, Y, a_prev, parameters, vocab_size)
        gradients, a_prev = rnn_backward(X, Y, parameters, cache_tuple, vocab_size)
        gradients = clip(gradients, 5)
        parameters = update_parameters(parameters, gradients, lr)
        
        if i % print_every == 0:
            print(f"Iteration {i}, Loss: {loss:.4f}")
            sentence, details = generate_text_with_probabilities(
                parameters, word_to_idx, idx_to_word,
                seed_word="i", length=6, top_k=3
            )
            print("Sample:", sentence)
            print("Step probabilities:")
            for d in details:
                print(d)
            print("-" * 60)
    
    return parameters, word_to_idx, idx_to_word

In [ ]:
text = "i like machine learning . i like deep learning . i like neural networks ."
tokens = text.split()

parameters, word_to_idx, idx_to_word = train_rnn_word_level(
    tokens,
    n_a=64,
    lr=0.1,
    num_iterations=10000,
    print_every=300
)

In [20]:
def print_generation_details(details):
    for d in details:
        print("=" * 50)
        print(f"Step {d['step']}")
        print(f"Input word      : {d['input_word']}")
        print(f"Predicted word  : {d['predicted_word']}")
        print(f"Predicted prob  : {d['predicted_prob']:.4f}")
        print("Top candidates  :")
        for word, prob in d["top_candidates"]:
            print(f"  - {word:12s}: {prob:.4f}")

In [21]:
sentence, details = generate_text_with_probabilities(
    parameters, word_to_idx, idx_to_word,
    seed_word="i", length=8, top_k=5
)

print("Sentence:", sentence)
print_generation_details(details)

Sentence: i like like neural . like like neural
Step 1
Input word      : i
Predicted word  : like
Predicted prob  : 0.9999
Top candidates  :
  - like        : 0.9999
  - machine     : 0.0001
  - i           : 0.0000
  - .           : 0.0000
  - learning    : 0.0000
Step 2
Input word      : like
Predicted word  : like
Predicted prob  : 1.0000
Top candidates  :
  - like        : 1.0000
  - machine     : 0.0000
  - i           : 0.0000
  - networks    : 0.0000
  - learning    : 0.0000
Step 3
Input word      : like
Predicted word  : neural
Predicted prob  : 0.9998
Top candidates  :
  - neural      : 0.9998
  - networks    : 0.0001
  - .           : 0.0000
  - i           : 0.0000
  - machine     : 0.0000
Step 4
Input word      : neural
Predicted word  : .
Predicted prob  : 0.9998
Top candidates  :
  - .           : 0.9998
  - i           : 0.0001
  - machine     : 0.0000
  - networks    : 0.0000
  - neural      : 0.0000
Step 5
Input word      : .
Predicted word  : like
Predicted prob  : 1.